# data_pipeline: Data pipelines for Pre-Training

## Learning Objectives
After studying and implementing this section, you should be able to:

1. Build a streaming data pipeline capable of tokenizing, chunking, shuffling, and batching terabytes of text without loading the entire dataset into memory.
2. Implement data quality filters such as deduplication, language detection, and content filtering, mirroring real-world pre-training pipelines.
3. Construct fixed-length training sequences, properly handling attention masks and document boundaries.
4. Measure and profile pipeline throughput to ensure the DataLoader feeds data fast enough to keep GPU compute fully utilized without starvation.


## What is the problem?
You have a tokenizer; now you need data.

This is not about a small dataset or a CSV file. To train a large language model, you need terabytes of text that has been cleaned, deduplicated, quality-filtered, and evaluated. Then, the text must be tokenized, divided into fixed-length sequences, and delivered to the model in random batches at sufficient speed. The pipeline's speed must be high enough that your eight-GPU cluster never has to wait for the next batch.

Many believe LLM training primarily depends on model architecture, but data plays a decisive role. Llama 3 was trained on 15.6 trillion tokens, GPT-3 on 300 billion tokens, and DeepSeek-V2 on 8.1 trillion tokens. The overall architecture of all three is more or less similar: multiple Transformer blocks stacked together, comprising attention and feed-forward layers. A significant portion of the difference in output quality among these models stems from the volume, composition, and quality of the training data.

DeepMind's Chinchilla paper elaborated on this. For any given compute budget, there is an optimal ratio between the number of model parameters and the number of training tokens. The research indicated that most models in 2022 were significantly undertrained, meaning they had too many parameters relative to the amount of data they were exposed to. For instance, a 70 billion parameter model trained on 1.4 trillion tokens, following the optimal Chinchilla ratio, outperformed Gopher, a 280 billion parameter model trained on only 300 billion tokens.

Therefore, your data pipeline determines whether the model truly learns language, knowledge, and useful patterns, or merely reproduces the noise present in the data.



In [2]:
import re
import hashlib
import random
import time
from collections import Counter, defaultdict
from typing import List, Tuple, Set, Dict, Any, Generator, Optional

In [ ]:
# basic clean text
def clean_text(text: str) -> str:
    """
    Cleans raw document text by stripping HTML tags, URLs, non-ASCII characters,
    and normalizing whitespace.

    Args:
        text (str): Input raw text document.

    Returns:
        str: Cleaned and normalized text string.
    """
    # TODO: Strip unwanted markup, non-ASCII noise, and normalize spaces/newlines.

    # if is NOT text-> return ""
    if not text:
        return ""

    # by regex remove tags
    text = re.sub(r"<[^>]+>", " ", text)

    # remove URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove ASCII
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # remove extera space in first and end
    text = re.sub(r"\s+", " ", text).strip()

    return text



Implementing this function in the pre-training pipeline simulates data quality filtering, screening out low-quality inputs such as promotional pages or spam content.

This function must evaluate three criteria:
- Word count
- Ratio of all-uppercase words
- Special character density

This function originates from the logic of well-known pipelines like RefinedWeb and Falcon, which check thresholds for length, uppercase ratio, and punctuation ratio for each document. These three criteria are aggregated to ensure each document passes through three gates.



In [ ]:
def quality_filter(
    text: str, 
    min_words: int = 50, 
    max_ratio_caps: int = 0.3, 
    max_ratio_special: float = 0.1
) -> bool:
    """
    Filters out low-quality documents based on length, capitalization ratio, and special character density.

    Args:
        text (str): Cleaned document text.
        min_words (int): Minimum required word count.
        max_ratio_caps (float): Maximum allowed ratio of ALL-CAPS words.
        max_ratio_special (float): Maximum allowed ratio of non-alphanumeric special characters.

    Returns:
        bool: True if the document meets quality criteria, False otherwise.
    """
    # TODO: Check word count thresholds and measure capitalization and special-character ratios.
    
    # min vocab - Word count
    words = text.split()
    if len(words) < min_words:
        return False

    # Ratio of all-uppercase words
    caps_words = sum(
        1 for w in words
        if w.isalpha() and w == w.upper()
    )
    if caps_words / len(words) > max_ratio_caps:
        return False

    # Special character density
    n_special = sum(1 for ch in text if not ch.isalnum())
    if n_special / len(text) > max_ratio_special:
        return False

    return True


This function serves as the foundation for the Deduplication algorithm (removing exact and near-duplicate documents) using MinHash LSH.

### Why Word-based Shingling (Word n-grams)?
- **What is a Shingle?** A continuous sequence of $k$ consecutive words (equivalent to a word-level $n$-gram with length $n = k$).
- **Why a Set?** In MinHash theory, a document is modeled as a "set" of shingles so that the Jaccard Similarity coefficient between two documents $A$ and $B$ can be computed:
  $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
- **Why Lowercase?** Variations in letter casing (such as "The" and "the") should not cause identical phrases to be treated as distinct; normalizing to `lower()` is essential.
- **Handling Short Texts:** If the document contains fewer than $k$ words, no shingle of length $k$ can be constructed; in this case, the function must return an empty set (`set()`).


### Implementation Steps
1. Convert the entire text to lowercase using `.lower()`.
2. Tokenize the text into a list of words using `.split()`.
3. Check the length condition: if `len(words) < k`, immediately return `set()`.
4. Iterate over the word list using a sliding window of length $k$, joining the tokens with a single space (`" ".join(...)`) to construct each shingle.
5. Collect and store all generated shingles in a unique `set`.


In [5]:
def get_shingles(text: str, k: int = 5) -> Set[str]:
    """
    Extracts k-shingles (word n-grams) from a text string.

    Args:
        text (str): Input document text.
        k (int): Size of word shingle (n-gram length).

    Returns:
        Set[str]: Unique set of k-word shingles.
    """
    # TODO: Lowercase, tokenize into words, and construct word n-gram shingles.
    if not text:
        return set()

    # Lower
    words = text.lower().split()

    # if k is graten than len(tex) -> cannot use
    if len(words) < k:
        return set()

    # Join
    shingles = {
        " ".join(words[i : i + k]) 
        for i in range(len(words) - k + 1)
    }

    return shingles


This function computes the MinHash signature (a fixed-length vector of size `num_hashes`) for a given set of shingles.

### MinHash Theory:
According to the MinHash theorem, the probability that the minimum hash value of two sets is equal under a random hash function $h_i$ is exactly equal to their Jaccard Similarity:

$$P(h_i(A) = h_i(B)) = J(A, B)$$

Therefore, given $m$ independent hash functions ($h_0, h_1, \dots, h_{m-1}$), the signature vector is constructed as follows:

$$\text{sig}[i] = \min_{s \in \text{shingles}} h_i(s)$$





In [6]:

# shingles from def get_shingles


def minhash_signature(shingles: Set[str], num_hashes: int = 128) -> List[int]:
    """
    Computes MinHash signature for a set of shingles using hash function permutations.

    Args:
        shingles (Set[str]): Set of unique word shingles.
        num_hashes (int): Length of the MinHash signature vector.

    Returns:
        List[int]: MinHash signature list of length (num_hashes,).
    """
    # TODO: Generate hash permutations for each seed and compute the minimum hash value per seed.

    if not shingles:
        return [0] * num_hashes

    signature = []

    for seed in range(num_hashes):
        min_val = float("inf")
        seed_bytes = str(seed).encode("utf-8")

        for s in shingles:
            
            # Combining a seed and a shingle to simulate an independent and deterministic hash function.
            h = hashlib.md5(seed_bytes + b"_" + s.encode("utf-8")).hexdigest()

            #Converting to an integer and optimizing processing speed
            
            val = int(h[:16], 16)

            if val < min_val:
                min_val = val

        signature.append(min_val)

    return signature


If we have $N$ documents, pairwise comparison of MinHash signatures requires $\frac{N(N-1)}{2}$ comparisons, resulting in a time complexity of $O(N^2)$.
The objective of LSH (Locality-Sensitive Hashing) is to reduce these comparisons: it maps documents with a high probability of similarity into a shared bucket, ensuring that only documents within the same bucket are compared.
A document's full signature consists of $M$ hashes (e.g., 128).

- If we enforce that "two documents must match across all 128 hashes," the condition becomes overly strict (retrieving only 100% identical copies).
- If we evaluate "each hash individually," the condition becomes overly relaxed, leading to a high number of dissimilar candidates (False Positives).

**LSH Mathematical Solution:** The signature is divided into $b$ bands, each containing $r$ numbers ($b \times r = M$).

The probability that two documents with Jaccard similarity $s$ become candidate pairs in at least one band is given by:

$$P(\text{Candidate}) = 1 - (1 - s^r)^b$$

This formula produces an S-curve that acts as a sharp threshold filter, identifying documents above the similarity threshold with high probability.

### Why convert to string and then hash with MD5?
```python
        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()
```

1. **Exact Matching Within a Band:** We want two documents to be assigned to the same bucket in a band if and only if all $r$ numbers in that band are strictly identical and in the exact same order.
2. **Fixed Size & Dictionary Key:** The sub-vector `chunk` is a Python list of integers. To look up this sub-vector in structures like `dict` or hash tables with $O(1)$ speed while minimizing memory overhead, we map it to a fixed-length hash string (32 Hex characters).


In [8]:
# signature from def minhash_signature

def lsh_buckets(signature: List[int], bands: int = 16) -> List[Tuple[int, str]]:
    """
    Divides MinHash signature into bands and maps each band to a bucket identifier using LSH.

    Args:
        signature (List[int]): MinHash signature vector of shape (num_hashes,).
        bands (int): Number of bands to divide the signature into.

    Returns:
        List[Tuple[int, str]]: List of (band_id, bucket_hash) tuples of length (bands,).
    """
    # TODO: Slice signature into bands, hash each band chunk, and map to bucket identifiers.
    num_hashes = len(signature)
    r = num_hashes // bands

    buckets = []
    for band_id in range(bands):

        chunk = signature[band_id * r : (band_id + 1) * r]

        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()

        buckets.append((band_id, bucket_hash))

    return buckets